# 05 — Transcripts

Reads `reference/transcript_coverage.csv` and `data/interim/transcript_segments.parquet` with their provenance, all produced by `scripts/05_build_transcript_inventory.py`, and asks which filers the pilot can actually sample from.

Every other reference table describes filings. This one describes the other side of the task: the calls the claims are made on. A filer with immaculate filing coverage and no transcript contributes nothing.

The transcripts themselves are third-party content and are not committed. What is committed is the coverage table and the character offsets where each call divides.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

REF = PROJECT_ROOT / "reference"
# keep_default_na: cik, continuous and match_method are empty for the ordinary
# case, and pandas would read those as NaN, breaking every == "" test below.
coverage = pd.read_csv(REF / "transcript_coverage.csv", keep_default_na=False)
segments = pd.read_parquet(PROJECT_ROOT / "data" / "interim" / "transcript_segments.parquet")
prov = json.loads((REF / "transcript_coverage.provenance.json").read_text())
counts = prov["counts"]

print(f"{counts['corpus_calls']:,} calls across {counts['corpus_symbols']} symbols, commit {prov['commit']}")
print(f"study window {prov['study_window'][0]}-{prov['study_window'][1]}")
print(f"{counts['study_filers_matched']} of {counts['study_filers']} study filers have transcripts")

33,362 calls across 685 symbols, commit 2020624
study window 2012-2024
121 of 150 study filers have transcripts


## Joining a ticker corpus to a filing study

The corpus is keyed by ticker and the study is keyed by CIK, and that join is less mechanical than it looks. A ticker is not stable over thirteen years: it moves when a company reorganises, it is reassigned after a delisting, and the SEC's public ticker file lists only current registrants.

Exxon Mobil is the case that makes the point. The study holds CIK 34088, taken from a 2023 revenue frame. The SEC's current ticker file maps XOM to CIK 2115436, a later holding entity, and the submissions record for 34088 carries no ticker at all. Joining on the public ticker file would have silently dropped the seventh largest filer in the study.

So the match runs on tickers from each filer's own EDGAR submissions record, with a short list of aliases for the cases where EDGAR and the corpus disagree about identity. Each alias was checked against the corpus's own company names.

In [2]:
for method, n in counts["match_methods"].items():
    print(f"  {method:<22} {n:>3} filers")
print()
for alias in prov["aliases_used"]:
    print(f"{alias['symbol']:<6} cik {alias['cik']:<9} {alias['name']}")
    print(f"       {alias['why']}\n")

  edgar_ticker           118 filers
  absent_from_corpus      24 filers
  verified_alias           3 filers
  not_listed               5 filers

XOM    cik 34088     EXXON MOBIL CORP
       The study holds the predecessor CIK from a 2023 revenue frame; the SEC ticker file now maps XOM to a later holding entity, and 34088 carries no ticker. The corpus rows under XOM name Exxon Mobil Corporation.

WBA    cik 1618921   Walgreens Boots Alliance, Inc.
       Delisted, so it has been removed from the SEC's current-registrant ticker file and its submissions record carries no ticker. The corpus rows under WBA name Walgreens Boots Alliance, Inc.

PARA   cik 813828    Paramount Global
       Succeeded by Paramount Skydance, so the submissions record for the predecessor carries no ticker. The corpus rows under PARA name Paramount Global.



Three aliases, and no fuzzy matching. That restraint is deliberate: an earlier attempt at matching on company names mapped Metropolitan Life onto 3M and Flex onto F5 Networks, because a shared prefix is not evidence of identity. Both would have been invisible in a coverage count and wrong in every claim adjudicated afterwards.

## The filers with no transcript

In [3]:
unmatched = pd.DataFrame(prov["unmatched_study_filers"])
unmatched["edgar_tickers"] = unmatched["edgar_tickers"].apply(lambda t: ",".join(t) if t else "")
print(unmatched["reason"].value_counts().to_string())
print()
unmatched.sort_values(["reason", "name"])[["reason", "name", "edgar_tickers"]].reset_index(drop=True)

reason
absent_from_corpus    24
not_listed             5



,reason,name,edgar_tickers
0,absent_from_corpus,"ARROW ELECTRONICS, INC.",ARW
1,absent_from_corpus,AVNET INC,AVT
2,absent_from_corpus,"Albertsons Companies, Inc.",ACI
3,absent_from_corpus,Athene Holding Ltd.,"ATH-PA,ATH-PB,ATH-PD,ATH-PE,ATHS"
4,absent_from_corpus,BERKSHIRE HATHAWAY INC,"BRK-B,BRK-A"
5,absent_from_corpus,CHS INC,"CHSCP,CHSCL,CHSCM,CHSCN,CHSCO"
6,absent_from_corpus,ENTERPRISE PRODUCTS PARTNERS L.P.,"EPD,EPDU"
7,absent_from_corpus,Energy Transfer LP,"ET,ET-PI"
8,absent_from_corpus,FLEX LTD.,FLEX
9,absent_from_corpus,Ferguson Enterprises Inc. /DE/,FERG


The two reasons are different problems and only one is about the corpus.

Five filers have no ticker in EDGAR at all, and none of them holds a public earnings call: Publix is employee-owned, CCO Holdings and PBF Holding file because they carry public debt, Metropolitan Life is a subsidiary of a listed parent, Berkshire Hathaway Energy likewise. Notebook 02 predicted exactly this when it found six corporate families holding twelve of the 150 slots, and said the transcript intersection would remove the debt-issuing subsidiaries without a new rule. It did.

Twenty-four filers are listed, have tickers, and are simply not in the corpus. Arrow Electronics, Avnet, Flex and US Foods are S&P 500 members, so this corpus is a 685-ticker subset rather than a complete index history. The partnerships among them, Energy Transfer, Enterprise Products and both Plains entities, are a separate matter: MLPs hold calls but are outside the index the corpus was built from.

## Coverage across the window

A corpus advertised as spanning twenty years says nothing about whether any one company is covered every quarter, and continuity is what a temporal split needs.

In [4]:
study = coverage[coverage["cik"] != ""].copy()
window = (prov["study_window"][1] - prov["study_window"][0] + 1) * 4

bands = [(window, window, f"all {window}"), (48, 51, "48-51"), (40, 47, "40-47"),
         (28, 39, "28-39"), (1, 27, "1-27")]
print(f"study filers by quarters covered of {window}:")
for lo, hi, label in bands:
    print(f"  {label:<8} {study['quarters_present'].between(lo, hi).sum():>3}")

print(f"\nfilers covering at least N of {window} quarters:")
for n in (52, 50, 48, 44, 40):
    print(f"  >= {n:>2}  {(study['quarters_present'] >= n).sum():>3}")

print(f"\nfilers whose coverage has no interior gap: {(study['gaps'] == 0).sum()} of {len(study)}")

study filers by quarters covered of 52:
  all 52    86
  48-51     14
  40-47      8
  28-39      7
  1-27       6

filers covering at least N of 52 quarters:
  >= 52   86
  >= 50   94
  >= 48  100
  >= 44  106
  >= 40  108

filers whose coverage has no interior gap: 106 of 121


**86 of 121 cover every quarter**, against a design that assumes 120 to 150 filers with continuous coverage. That gap is the finding.

The shape of the shortfall matters more than its size. 106 of the 121 have no interior gap at all: their coverage is contiguous and simply starts late or ends early, because the company listed after 2012 or was acquired before 2024. An interior gap breaks a temporal split; a short span only shortens it.

So the choice is between a smaller study set, a shorter window, or a looser definition of continuous. The third is the cheapest and the numbers say so.

In [5]:
calls = segments[segments["symbol"].isin(study["symbol"])]
def covered(frame, start, end):
    """Quarters covered per symbol inside a window."""
    w = frame[frame["year"].between(start, end)]
    return w.groupby("symbol")[["year", "quarter"]].apply(lambda g: len(g.drop_duplicates()))

print("filers covering every quarter, by window:")
for start in (2012, 2013, 2014, 2015, 2016):
    need = (2024 - start + 1) * 4
    print(f"  {start}-2024  ({need:>2} quarters): {(covered(calls, start, 2024) == need).sum():>3} filers")

print("\nfilers over the full 2012-2024 window, allowing a few missing quarters:")
full = covered(calls, 2012, 2024)
for miss in (0, 1, 2, 4, 8):
    print(f"  at most {miss} missing of 52: {(full >= 52 - miss).sum():>3} filers")

# Same count is not the same filers, so compare the sets rather than the totals.
short_window = set(covered(calls, 2016, 2024).pipe(lambda c: c[c == 36]).index)
loose_full = set(full[full >= 48].index)
print(f"\nboth options keep {len(short_window)} filers, and they are not the same {len(short_window)}:")
print(f"  only in 2016-2024 continuous:      {sorted(short_window - loose_full)}")
print(f"  only in <=4 missing over 2012-2024: {sorted(loose_full - short_window)}")

filers covering every quarter, by window:
  2012-2024  (52 quarters):  86 filers
  2013-2024  (48 quarters):  94 filers
  2014-2024  (44 quarters):  94 filers
  2015-2024  (40 quarters):  96 filers
  2016-2024  (36 quarters): 100 filers

filers over the full 2012-2024 window, allowing a few missing quarters:
  at most 0 missing of 52:  86 filers
  at most 1 missing of 52:  91 filers
  at most 2 missing of 52:  94 filers
  at most 4 missing of 52: 100 filers
  at most 8 missing of 52: 106 filers

both options keep 100 filers, and they are not the same 100:
  only in 2016-2024 continuous:      ['CHTR', 'CNC', 'HCA', 'HPE', 'PYPL', 'UAL']
  only in <=4 missing over 2012-2024: ['CI', 'DG', 'GD', 'LOW', 'MPC', 'TGT']


Both options land on 100 filers, and they are not the same 100. Six are continuous from 2016 but miss too much of the earlier window, because they listed or spun out after 2012: Charter, Centene, HCA, Hewlett Packard Enterprise, PayPal and United Airlines. Six others clear the loosened threshold over the full window while carrying a gap inside 2016 to 2024: Cigna, Dollar General, General Dynamics, Lowe's, Marathon Petroleum and Target.

So the choice is not a free win either way. Loosening the threshold buys sixteen extra quarters of claims from every filer it keeps; shortening the window buys six companies that did not exist as filers at the start of it. Which matters depends on whether the study wants length or breadth, and that is a design decision rather than a property of this table, so the table records quarters and gaps per filer and leaves the threshold to whoever samples.

## Prepared remarks against Q&A

The two halves of a call are different objects. Prepared remarks are written and reviewed; the Q&A is unscripted and is where management gets pushed into saying something specific. No field marks the boundary, so it is found in the text by the operator handing over to the first analyst.

In [6]:
print(f"{len(segments):,} calls\n")
order = ["ok", "implausible_position", "not_found", "empty"]
for label in order:
    n = (segments["confidence"] == label).sum()
    if n:
        print(f"  {label:<22} {n:>6,}  {100*n/len(segments):>5.1f}%")
failed = (segments["confidence"] != "ok").sum()
print(f"\nfailure rate: {100*failed/len(segments):.1f}%\n")
print("marker that found the boundary:")
print(segments["marker"].fillna("none").value_counts().to_string())

33,362 calls

  ok                     31,404   94.1%
  implausible_position      543    1.6%
  not_found               1,289    3.9%
  empty                     126    0.4%

failure rate: 5.9%

marker that found the boundary:
marker
first_question_intro    25519
first_question           6008
none                     1415
open_for_questions        304
now_begin_questions       116


94.1 per cent split cleanly, so the failure rate is 5.9. Four fifths of the boundaries come from the operator's explicit handover; the bare "first question" below it catches most of the rest, and is ranked lower because an operator can mention questions in the opening remarks.

Which is why position is checked as well as pattern. A match is only accepted where it lands between 5 and 90 per cent of the transcript.

In [7]:
ok = segments.dropna(subset=["split_fraction"])
band = prov["qa_split_band"]
print(f"where the boundary lands, as a fraction of the transcript:")
for p in (1, 5, 25, 50, 75, 95, 99):
    print(f"  p{p:<3} {ok['split_fraction'].quantile(p/100):.3f}")
print(f"\nband {band[0]} to {band[1]}: "
      f"{((ok['split_fraction'] < band[0]) | (ok['split_fraction'] > band[1])).sum():,} outside")

# The failure rate is not constant: early transcripts are less structured.
by_year = segments.groupby("year").apply(
    lambda g: 100 * (g["confidence"] != "ok").mean(), include_groups=False
).round(1)
print("\nfailure rate by year:")
print(by_year.loc[2005:2024].to_string())

where the boundary lands, as a fraction of the transcript:
  p1   0.045
  p5   0.131
  p25  0.308
  p50  0.391
  p75  0.480
  p95  0.629
  p99  0.828

band 0.05 to 0.9: 543 outside

failure rate by year:
year
2005    38.8
2006    16.5
2007    16.4
2008     6.2
2009     5.1
2010     6.7
2011     6.3
2012     7.4
2013     5.7
2014     5.4
2015     5.6
2016     6.2
2017     6.1
2018     4.1
2019     4.4
2020     4.1
2021     4.5
2022     4.9
2023     4.4
2024     5.8


The median call turns to questions 39 per cent of the way through, and the band only rejects the tails: a split at 2 per cent leaves no prepared remarks, one at 95 per cent leaves no Q&A.

The failure rate is not uniform across the corpus. It is 39 per cent in 2005 and 16 per cent through 2007, then settles between 4 and 7 per cent from 2008 onward, because early transcripts carry less structure. The study window starts in 2012 for an unrelated reason, XBRL tagging, and that choice happens to avoid the worst of this too.

## Licence

In [8]:
raw_prov = json.loads((PROJECT_ROOT / "data" / "raw" / "transcripts.provenance.json").read_text())
print(f"source: {raw_prov['source']['repo']}")
print(f"derived from: {', '.join(raw_prov['source']['derived_from'])}\n")
for line in raw_prov["licence_position"]:
    print(f"- {line}")

source: RudrakshNanavaty/earnings-call-data
derived from: Bose345/sp500_earnings_transcripts, kurry/sp500_earnings_transcripts

- The release is tagged MIT, as are Bose345/sp500_earnings_transcripts and kurry/sp500_earnings_transcripts beneath it.
- None of the three names the vendor the transcripts originally came from, and several transcripts carry a TRANSCRIPT SPONSOR line, which is a vendor artefact.
- An MIT tag applied by an uploader does not grant rights the uploader does not hold, so the text is treated as third-party content.
- The corpus is pulled, never committed, and never redistributed. Released artifacts carry character offsets and labels plus a script that rebuilds the text from this source.


The chain is three HuggingFace datasets deep and every link is tagged MIT, but none of them names the vendor the transcripts came from, and some transcripts still carry a `TRANSCRIPT SPONSOR` line, which is a vendor artefact. One card tags MIT and then adds "for research and educational use only", which MIT does not say.

So the tag is an uploader's assertion rather than a licence from a copyright holder, and the study does not rely on it. The corpus is pulled and never committed, and a released dataset carries character offsets and labels plus a script that rebuilds the text from this source. That was the project's stated position before this pull; the licence chain confirms it rather than changing it.

## What this implies for the pilot

**The study set is 121 filers, not 150.** Twenty-nine have no transcript in this corpus. Five of those hold no public call at all, which is structural and confirms what the filer selection notebook predicted. The other 24 are listed companies the corpus does not carry, including four S&P 500 members, so the corpus is a subset rather than an index history.

**Continuous coverage is 86 filers, against a design that assumes 120 to 150.** The design's target cannot be met from this corpus. Loosening "continuous" to at most four missing quarters gives 100 filers over the full window, which beats shortening the window to 2016 for the same count. The table carries quarters and gaps per filer so the threshold stays a study decision.

**The Q&A split works on 94 per cent of calls** and better than that inside the study window. The 5.9 per cent that fail are recorded with the reason rather than dropped, so a pass that needs only prepared remarks can use the split and a pass that needs certainty can filter on confidence.

**The join is the fragile part.** Ticker to CIK is not stable across thirteen years, and one wrong match is worse than one missing filer: a missing filer is a smaller sample, a wrong match adjudicates one company's claims against another's filings. Every match here carries the method that produced it, and the three aliases carry their justification.